# Etapa 1: Fundamentos, Desbalance y Evaluación de SMOTE

**Objetivo de Negocio :**
Evaluar el impacto de técnicas de oversampling (SMOTE) en la detección de fraudes con tarjetas de crédito. 
Dado que un BPO de revisión de fraudes tiene un costo operativo por cada "Falso Positivo" (horas hombre revisando transacciones legítimas), el modelo debe priorizar una alta Precisión sin sacrificar excesivamente el Recall.

**Hipótesis:**
SMOTE es vital para modelos lineales simples ante desbalances extremos, pero en algoritmos de ensamble (Random Forest / XGBoost) inyecta ruido que degrada la Precisión y satura la operación del BPO.

In [8]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
from imblearn.over_sampling import SMOTE
import warnings
warnings.filterwarnings('ignore')

# Carga de los datos crudos (inmutables)
df = pd.read_csv('../data/raw/creditcard.csv')

# Visualización rápida del desbalance masivo
clases = df['Class'].value_counts()
print(f"Transacciones Legítimas (0): {clases[0]}")
print(f"Transacciones Fraudulentas (1): {clases[1]}")
print(f"Porcentaje de Fraude: {(clases[1]/len(df))*100:.3f}%")

Transacciones Legítimas (0): 284315
Transacciones Fraudulentas (1): 492
Porcentaje de Fraude: 0.173%


In [9]:
# Separar variables predictoras (X) de la etiqueta (y)
X = df.drop('Class', axis=1)
y = df['Class']

# Regla de Oro: Separar Train/Test ANTES de cualquier balanceo
# stratify=y asegura que el 0.17% de fraude se mantenga en ambos sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Datos para entrenar: {X_train.shape[0]}")
print(f"Datos para validar: {X_test.shape[0]}")

Datos para entrenar: 227845
Datos para validar: 56962


In [7]:
print("="*60)
print(" EXPERIMENTO 1: RANDOM FOREST (Ensamblado Complejo)")
print("="*60)

# 1. Random Forest Base (Datos Originales)
rf_base = RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1)
rf_base.fit(X_train, y_train)
y_pred_base = rf_base.predict(X_test)

print("\n[Modelo Base - Sin SMOTE]")
print(classification_report(y_test, y_pred_base, target_names=['Legítima', 'Fraude']))

# 2. Aplicar SMOTE solo al Train
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

# 3. Random Forest con SMOTE
rf_smote = RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1)
rf_smote.fit(X_train_smote, y_train_smote)
y_pred_smote = rf_smote.predict(X_test)

print("\n[Modelo Alterado - Con SMOTE]")
print(classification_report(y_test, y_pred_smote, target_names=['Legítima', 'Fraude']))

 EXPERIMENTO 1: RANDOM FOREST (Ensamblado Complejo)

[Modelo Base - Sin SMOTE]
              precision    recall  f1-score   support

    Legítima       1.00      1.00      1.00     56864
      Fraude       0.94      0.82      0.87        98

    accuracy                           1.00     56962
   macro avg       0.97      0.91      0.94     56962
weighted avg       1.00      1.00      1.00     56962


[Modelo Alterado - Con SMOTE]
              precision    recall  f1-score   support

    Legítima       1.00      1.00      1.00     56864
      Fraude       0.83      0.83      0.83        98

    accuracy                           1.00     56962
   macro avg       0.91      0.91      0.91     56962
weighted avg       1.00      1.00      1.00     56962



In [10]:
print("="*60)
print(" EXPERIMENTO 2: REGRESIÓN LOGÍSTICA (Lineal Simple)")
print("="*60)

# 1. Logística Base (El modelo colapsa)
lr_base = LogisticRegression(max_iter=1000, random_state=42)
lr_base.fit(X_train, y_train)
print("\n[Logística Base - Sin SMOTE]")
print(classification_report(y_test, lr_base.predict(X_test), target_names=['Legítima', 'Fraude']))

# 2. Logística Salvada (SMOTE rescata al modelo)
lr_smote = LogisticRegression(max_iter=1000, random_state=42)
lr_smote.fit(X_train_smote, y_train_smote)
print("\n[Logística Rescatada - Con SMOTE]")
print(classification_report(y_test, lr_smote.predict(X_test), target_names=['Legítima', 'Fraude']))

 EXPERIMENTO 2: REGRESIÓN LOGÍSTICA (Lineal Simple)

[Logística Base - Sin SMOTE]
              precision    recall  f1-score   support

    Legítima       1.00      1.00      1.00     56864
      Fraude       0.83      0.68      0.75        98

    accuracy                           1.00     56962
   macro avg       0.91      0.84      0.87     56962
weighted avg       1.00      1.00      1.00     56962


[Logística Rescatada - Con SMOTE]
              precision    recall  f1-score   support

    Legítima       1.00      0.99      0.99     56864
      Fraude       0.12      0.90      0.21        98

    accuracy                           0.99     56962
   macro avg       0.56      0.94      0.60     56962
weighted avg       1.00      0.99      0.99     56962



## Conclusión de la Etapa 1
1. **Modelos Lineales:** Requieren obligatoriamente SMOTE u otra técnica de balanceo para no ignorar la clase minoritaria.
2. **Modelos de Ensamble (Bosques/XGBoost):** Manejan el desbalance extremo de forma natural aislando nodos. Aplicarles SMOTE en este escenario específico destruye la Precisión, aumentando drásticamente los Falsos Positivos.
3. **Decisión Arquitectónica:** Se descarta el uso de SMOTE para las siguientes etapas. Se procederá a la **Etapa 2** construyendo variables complejas (Feature Engineering) sobre los datos desbalanceados originales, preparándolos para alimentar un modelo XGBoost con optimización de pesos (Scale Pos Weight).